# Module 8 Exercise: KV-cache benchmark for autoregressive generation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nsteve2407/llm-transformers-course/blob/master/notebooks/08-efficient-transformers/exercise_starter.ipynb)

Module page: [Module 8: Efficient Transformers](https://nsteve2407.github.io/llm-transformers-course/modules/08-efficient-transformers/)

Autoregressive decoding generates one token at a time, and each new token's forward pass can either **recompute
attention over the entire sequence-so-far** (the naive approach) or **reuse the key/value projections already
computed for earlier positions** via a KV cache. This notebook makes that difference concrete and measurable
with a real pretrained GPT-2 small:

1. Implement a no-cache generation loop (`use_cache=False`, full forward pass every step -- O(seq_len) work
   per step, O(seq_len^2) total).
2. Implement a with-cache generation loop (`use_cache=True`, `past_key_values` reused, only the newest token
   forwarded each step -- O(1) work per step after prefill, O(seq_len) total).
3. Benchmark both across a sweep of generation lengths and plot wall-clock time and peak GPU memory.
4. Verify the textbook KV-cache memory formula (`2 * batch * seq_len * layers * heads * head_dim * bytes`)
   against what's actually measured on the GPU.
5. Extend the formula to hypothetical MQA and GQA configurations and quantify the memory savings that motivate
   them.

Unlike Module 5's exercise, `SMOKE_TEST` here does **not** swap in a randomly-initialized model -- the whole
point of this notebook is measuring *real* inference timing/memory characteristics, and there's no meaningful
"random weights" analogue for a speed/memory benchmark. `SMOKE_TEST=1` instead shrinks the generation-length
sweep and the benchmark batch size, so the notebook still finishes quickly while exercising the identical code
path against real GPT-2 weights.

In [ ]:
try:
    import transformers
except ImportError:
    %pip install -q transformers

In [ ]:
import os
import time

import torch
import matplotlib.pyplot as plt

from transformers import GPT2LMHeadModel, GPT2Tokenizer

SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
HAS_CUDA = device == "cuda"
print(f"SMOKE_TEST={SMOKE_TEST}, device={device}")

## Design choices and judgment calls (documented up front)

- **`SMOKE_TEST` scope**: reduces `GENERATION_LENGTHS` (the sweep of how many new tokens to generate) and
  `BENCHMARK_BATCH_SIZE` only. GPT-2 small (~124M params, ~500MB fp32 / ~250MB fp16) is always loaded with its
  real pretrained weights in both branches -- there's no "shape-only" analogue for a timing/memory benchmark,
  unlike Module 5 where random weights were fine because only architectural mechanics were being checked.
- **Precision**: fp16 on CUDA (halves activation and KV-cache memory, and is how these models are actually
  served in practice), fp32 on CPU (fp16 matmuls are not well-accelerated on CPU in this environment).
- **Greedy decoding throughout**: `argmax` at every step, no sampling. This keeps the no-cache and with-cache
  loops mathematically required to produce *identical* token sequences for the same prompt -- a strong
  correctness check (below) that a subtler sampling-based comparison wouldn't give us for free.
- **Timing method**: `torch.cuda.Event(enable_timing=True)` pairs (GPU-side, avoids CPU/GPU async-launch skew)
  when CUDA is available, `time.perf_counter()` otherwise.
- **Peak memory**: `torch.cuda.max_memory_allocated()` after `torch.cuda.reset_peak_memory_stats()`, CUDA-only
  by construction -- skipped gracefully (not crashed) on CPU-only environments.
- **Memory-formula comparison is a delta, not an absolute match**: the textbook formula
  `2 * batch * seq_len * layers * heads * head_dim * bytes_per_element` gives the *steady-state* size of the
  KV cache tensors themselves, but `max_memory_allocated()` also captures model weights (constant, cancels out
  in a delta) plus one large, specific *transient* effect: HuggingFace's default `past_key_values` grows via a
  non-in-place `torch.cat` at every decode step, so the previous (shorter) cache and the newly-concatenated
  (one token longer) cache are **both** alive in GPU memory simultaneously during that step. Peak measured
  memory therefore reflects roughly *2x* the steady-state cache size, not 1x -- see Part 5 for the measured
  ratio and the same mechanism explained in more detail. We isolate the cache's *own* footprint by comparing
  peak-memory *deltas* between two different generation lengths (the constant model-weight overhead cancels
  out), and expect the theoretical delta to land at roughly *half* the measured delta, not match it exactly.

## Setup: load GPT-2 small

In [ ]:
model_dtype = torch.float16 if HAS_CUDA else torch.float32

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

model = GPT2LMHeadModel.from_pretrained("gpt2", torch_dtype=model_dtype)
model = model.to(device).eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"gpt2 loaded: {n_params:,} parameters, dtype={model_dtype}, device={device}")
print(
    f"config: n_layer={model.config.n_layer}, n_head={model.config.n_head}, "
    f"n_embd={model.config.n_embd}"
)

## Part 1: generation without KV caching

At every step, re-run the full forward pass over the *entire* sequence generated so far
(`use_cache=False`), discard everything the model computed, and keep only the new token's logits. Step `t`
(0-indexed, prompt length `p`) does a forward pass over `p + t` tokens, so total work across `n` generated
tokens is `sum_{t=0}^{n-1} (p + t) = O(n^2)` for `n >> p` -- quadratic in the number of generated tokens.

In [ ]:
def generate_no_cache(model, input_ids, num_new_tokens):
    '''Greedy generation that recomputes the full forward pass over the whole sequence-so-far at every
    step (use_cache=False) -- the naive approach whose total cost is O(num_new_tokens^2).

    input_ids: (batch, prompt_len) LongTensor prompt.
    Returns: (batch, prompt_len + num_new_tokens) LongTensor.
    '''
    # TODO: for num_new_tokens steps: run model(input_ids=generated, use_cache=False) over the FULL
    # sequence generated so far, take argmax of logits[:, -1, :] as the next token, and torch.cat it onto
    # `generated` along dim=1. Wrap the whole loop in torch.no_grad(). Return the final `generated` tensor.
    raise NotImplementedError("TODO: implement the no-cache generation loop")

## Part 2: generation with KV caching

Prefill once over the prompt (`use_cache=True`), keep the returned `past_key_values`, and at every
subsequent step forward *only the newest token* -- its query attends over the cached keys/values from every
earlier position plus itself, so each step is O(1) work (independent of how long the sequence has gotten so
far) and total cost across `n` generated tokens is `O(n)`.

In [ ]:
def generate_with_cache(model, input_ids, num_new_tokens):
    '''Greedy generation using past_key_values: after the initial prefill over the full prompt, each
    subsequent step forwards only the single newest token, letting the model attend over cached
    keys/values for every earlier position. Total cost across num_new_tokens steps is O(num_new_tokens).

    input_ids: (batch, prompt_len) LongTensor prompt.
    Returns: (batch, prompt_len + num_new_tokens) LongTensor.
    '''
    # TODO: for num_new_tokens steps: call model(input_ids=next_input, past_key_values=past_key_values,
    # use_cache=True); save outputs.past_key_values for the next iteration; take argmax of
    # outputs.logits[:, -1, :] as the next token; torch.cat it onto `generated`; then set next_input to
    # JUST that new token (not the whole sequence) for the following step. The very first call should pass
    # the full prompt as next_input with past_key_values=None. Wrap the loop in torch.no_grad(). Return
    # the final `generated` tensor.
    raise NotImplementedError("TODO: implement the with-cache generation loop")

### Correctness check: both loops must produce identical tokens

Both loops are greedy (deterministic `argmax`) implementations of the *same* mathematical model, so for
the same prompt they must generate byte-for-byte identical continuations -- the cache changes *how* the
logits at each step are computed, not *what* they are. This is a much stronger check than "the code runs
without error": it directly catches, e.g., an accidentally-stale cache, a shifted position, or the
no-cache loop secretly benefiting from caching internally.

In [ ]:
check_prompt = "The quick brown fox jumps over the lazy dog and then"
check_input_ids = tokenizer(check_prompt, return_tensors="pt")["input_ids"].to(device)

no_cache_out = generate_no_cache(model, check_input_ids, num_new_tokens=10)
with_cache_out = generate_with_cache(model, check_input_ids, num_new_tokens=10)

assert torch.equal(no_cache_out, with_cache_out), "no-cache and with-cache generations diverged!"
print("no-cache and with-cache generations are identical, as expected for greedy decoding.")
print("no-cache continuation:  ", tokenizer.decode(no_cache_out[0, check_input_ids.shape[1]:]))
print("with-cache continuation:", tokenizer.decode(with_cache_out[0, check_input_ids.shape[1]:]))

## Part 3: benchmark harness

For each generation length in the sweep, time both approaches on an identical batch of prompts and (on
GPU) record peak memory. `SMOKE_TEST=1` shrinks both the sweep and the batch size so the full notebook
runs in well under a minute; the full sweep is representative of the quadratic-vs-linear contrast at
lengths large enough to see it clearly.

In [ ]:
GENERATION_LENGTHS = [32, 64, 128] if SMOKE_TEST else [32, 64, 128, 256, 512]
BENCHMARK_BATCH_SIZE = 2 if SMOKE_TEST else 8
PROMPT_LENGTH = 16  # tokens of fixed-length prompt prefix used for every benchmark run

_prompt_text = (
    "The history of artificial intelligence began with philosophers and mathematicians who dreamed of "
    "machines that could reason, calculate, and eventually think for themselves, long before any such "
    "machine actually existed."
)
_base_prompt_ids = tokenizer(_prompt_text, return_tensors="pt")["input_ids"][:, :PROMPT_LENGTH]
assert _base_prompt_ids.shape[1] == PROMPT_LENGTH, "prompt text too short for PROMPT_LENGTH"
benchmark_prompt_ids = _base_prompt_ids.repeat(BENCHMARK_BATCH_SIZE, 1).to(device)
print(f"GENERATION_LENGTHS={GENERATION_LENGTHS}, BENCHMARK_BATCH_SIZE={BENCHMARK_BATCH_SIZE}")


def measure_generation(gen_fn, input_ids, num_new_tokens):
    '''Time one full generation run of gen_fn and, on CUDA, record peak memory allocated during it.

    Returns (elapsed_seconds, peak_memory_bytes_or_None). peak_memory is None on CPU-only environments,
    since torch.cuda.max_memory_allocated() is CUDA-only.
    '''
    if HAS_CUDA:
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        start_event = torch.cuda.Event(enable_timing=True)
        end_event = torch.cuda.Event(enable_timing=True)
        start_event.record()
        gen_fn(model, input_ids, num_new_tokens)
        end_event.record()
        torch.cuda.synchronize()
        elapsed = start_event.elapsed_time(end_event) / 1000.0  # ms -> s
        peak_memory = torch.cuda.max_memory_allocated()
    else:
        start = time.perf_counter()
        gen_fn(model, input_ids, num_new_tokens)
        elapsed = time.perf_counter() - start
        peak_memory = None
    return elapsed, peak_memory

In [ ]:
# Warm up CUDA kernels / allocator once, outside the timed sweep -- for BOTH generation paths, so neither
# gets an unfair one-time first-call cost (cuDNN/cuBLAS algorithm selection, allocator growth) folded into
# its timed measurements below.
if HAS_CUDA:
    measure_generation(generate_no_cache, benchmark_prompt_ids, 4)
    measure_generation(generate_with_cache, benchmark_prompt_ids, 4)

results = {"length": [], "no_cache_time": [], "cache_time": [], "no_cache_mem": [], "cache_mem": []}
for length in GENERATION_LENGTHS:
    no_cache_time, no_cache_mem = measure_generation(generate_no_cache, benchmark_prompt_ids, length)
    cache_time, cache_mem = measure_generation(generate_with_cache, benchmark_prompt_ids, length)
    results["length"].append(length)
    results["no_cache_time"].append(no_cache_time)
    results["cache_time"].append(cache_time)
    results["no_cache_mem"].append(no_cache_mem)
    results["cache_mem"].append(cache_mem)

    speedup = no_cache_time / cache_time
    line = (
        f"length={length:4d}  no-cache={no_cache_time:8.4f}s  with-cache={cache_time:8.4f}s  "
        f"speedup={speedup:5.2f}x"
    )
    if HAS_CUDA:
        line += f"  no-cache-mem={no_cache_mem / 1e6:7.1f}MB  cache-mem={cache_mem / 1e6:7.1f}MB"
    print(line)

print()
print(
    f"Speedup grew from {results['no_cache_time'][0] / results['cache_time'][0]:.2f}x at length="
    f"{results['length'][0]} to {results['no_cache_time'][-1] / results['cache_time'][-1]:.2f}x at "
    f"length={results['length'][-1]} -- exactly the widening gap expected from O(n^2) vs O(n) scaling."
)

## Part 4: plotting time and memory vs. generation length

In [ ]:
fig, axes = plt.subplots(1, 2 if HAS_CUDA else 1, figsize=(12, 4.5) if HAS_CUDA else (6, 4.5))
time_ax = axes[0] if HAS_CUDA else axes

time_ax.plot(results["length"], results["no_cache_time"], marker="o", label="no cache (use_cache=False)")
time_ax.plot(results["length"], results["cache_time"], marker="o", label="with cache (use_cache=True)")
time_ax.set_xlabel("generation length (new tokens)")
time_ax.set_ylabel("wall-clock time (s)")
time_ax.set_title("Generation time: no-cache (~quadratic) vs. cached (~linear)")
time_ax.legend()
time_ax.grid(alpha=0.3)

if HAS_CUDA:
    mem_ax = axes[1]
    mem_ax.plot(results["length"], [m / 1e6 for m in results["no_cache_mem"]], marker="o", label="no cache")
    mem_ax.plot(results["length"], [m / 1e6 for m in results["cache_mem"]], marker="o", label="with cache")
    mem_ax.set_xlabel("generation length (new tokens)")
    mem_ax.set_ylabel("peak GPU memory (MB)")
    mem_ax.set_title("Peak GPU memory vs. generation length")
    mem_ax.legend()
    mem_ax.grid(alpha=0.3)
else:
    print("CUDA not available: skipping peak-memory plot (torch.cuda.max_memory_allocated is CUDA-only).")

plt.tight_layout()
plt.show()

## Part 5: verifying the KV-cache memory formula

The textbook KV-cache size (in bytes, for the cached keys+values across all layers) is:

```
2 * batch * seq_len * num_layers * num_heads * head_dim * bytes_per_element
```

The leading `2` is keys *and* values; `head_dim = n_embd / n_head`. We compute this for GPT-2's actual
config and compare a *delta* between two sweep lengths against the *measured* peak-memory delta between
those same two with-cache runs -- see the design-choices note above for why a delta (not an absolute
match) is the right comparison.

In [ ]:
def theoretical_kv_cache_bytes(batch, seq_len, num_layers, num_kv_heads, head_dim, bytes_per_element):
    return 2 * batch * seq_len * num_layers * num_kv_heads * head_dim * bytes_per_element


head_dim = model.config.n_embd // model.config.n_head
bytes_per_element = torch.tensor([], dtype=model_dtype).element_size()
print(f"head_dim={head_dim}, bytes_per_element={bytes_per_element} ({model_dtype})")

if HAS_CUDA and len(results["length"]) >= 2:
    short_idx, long_idx = 0, -1
    short_len = PROMPT_LENGTH + results["length"][short_idx]
    long_len = PROMPT_LENGTH + results["length"][long_idx]

    theoretical_short = theoretical_kv_cache_bytes(
        BENCHMARK_BATCH_SIZE, short_len, model.config.n_layer, model.config.n_head, head_dim, bytes_per_element
    )
    theoretical_long = theoretical_kv_cache_bytes(
        BENCHMARK_BATCH_SIZE, long_len, model.config.n_layer, model.config.n_head, head_dim, bytes_per_element
    )
    theoretical_delta = theoretical_long - theoretical_short
    measured_delta = results["cache_mem"][long_idx] - results["cache_mem"][short_idx]

    print(f"sequence length {short_len} -> {long_len} tokens (prompt + generated)")
    print(f"theoretical KV-cache delta: {theoretical_delta / 1e6:8.2f} MB")
    print(f"measured peak-memory delta: {measured_delta / 1e6:8.2f} MB")
    if measured_delta > 0:
        print(f"theoretical / measured ratio: {theoretical_delta / measured_delta:.2f}")
    print(
        "The theoretical figure lands at roughly HALF the measured delta, and that ~0.50 ratio has a "
        "precise, mechanical cause rather than a vague one: HuggingFace's default `past_key_values` grows "
        "via a non-in-place torch.cat at every decode step, so the previous (shorter) cache tensor and the "
        "newly-concatenated (one token longer) cache tensor are BOTH alive in GPU memory at the same time "
        "during that step. Peak measured memory therefore captures roughly 2x the steady-state cache size "
        "-- not 1x -- which is exactly why production inference engines (vLLM's paged attention, "
        "HuggingFace's `StaticCache`) preallocate the KV cache up front instead of growing it via "
        "concatenation: preallocation avoids ever holding two full copies of the cache in memory at once."
    )
else:
    print("Skipping measured-vs-theoretical comparison (needs CUDA and >= 2 sweep points).")
    example_len = PROMPT_LENGTH + GENERATION_LENGTHS[-1]
    theoretical_bytes = theoretical_kv_cache_bytes(
        BENCHMARK_BATCH_SIZE, example_len, model.config.n_layer, model.config.n_head, head_dim, bytes_per_element
    )
    print(
        f"theoretical KV-cache size at seq_len={example_len}: {theoretical_bytes / 1e6:.2f} MB "
        "(formula only -- no CUDA measurement available to compare against)"
    )

## Part 6: extension -- MQA and GQA

Multi-query attention (MQA) shares a *single* key/value head across all query heads; grouped-query
attention (GQA) shares each key/value head across a *group* of query heads, interpolating between full
multi-head attention (MHA, GPT-2's actual scheme) and MQA. Both leave `num_kv_heads` as the only changed
input to the memory formula -- everything else about the model is unchanged. This directly attacks the
KV-cache-size (and therefore memory-bandwidth) bottleneck the GQA paper (Ainslie et al., 2023) identifies
as the dominant cost of long-context autoregressive decoding.

In [ ]:
example_seq_len = PROMPT_LENGTH + GENERATION_LENGTHS[-1]
GQA_GROUPS = 4

mha_bytes = theoretical_kv_cache_bytes(
    1, example_seq_len, model.config.n_layer, model.config.n_head, head_dim, bytes_per_element
)
gqa_kv_heads = max(1, model.config.n_head // GQA_GROUPS)
gqa_bytes = theoretical_kv_cache_bytes(
    1, example_seq_len, model.config.n_layer, gqa_kv_heads, head_dim, bytes_per_element
)
mqa_bytes = theoretical_kv_cache_bytes(
    1, example_seq_len, model.config.n_layer, 1, head_dim, bytes_per_element
)

print(
    f"KV-cache size at seq_len={example_seq_len}, batch=1 (single-sequence decode -- the "
    "memory-bandwidth-bound regime):"
)
print(f"  MHA ({model.config.n_head} KV heads):             {mha_bytes / 1e6:7.2f} MB")
print(
    f"  GQA ({gqa_kv_heads} KV heads, {GQA_GROUPS} groups):  {gqa_bytes / 1e6:7.2f} MB  "
    f"({mha_bytes / gqa_bytes:.1f}x smaller than MHA)"
)
print(f"  MQA (1 KV head):                {mqa_bytes / 1e6:7.2f} MB  ({mha_bytes / mqa_bytes:.1f}x smaller than MHA)")

## Summary

- **No-cache generation re-runs the full forward pass every step**, so step `t` costs `O(prompt_len + t)`
  and total cost across `n` generated tokens is `O(n^2)`.
- **With-cache generation forwards only the newest token every step** after a one-time prefill, so each
  step costs `O(1)` (independent of how long the sequence has gotten) and total cost is `O(n)`. The
  benchmark above shows the speedup widening as generation length grows -- exactly the quadratic-vs-linear
  gap the two loops' complexity predicts.
- **That speed comes from memory, not for free**: the cache holds every past layer's keys and values, so
  its footprint grows linearly with sequence length (and with batch size, layers, heads, and head_dim) --
  the formula verified above. For long contexts or large batches, this cache -- not the model's weights --
  can become the dominant consumer of GPU memory.
- **MQA and GQA trade a little quality for a lot less KV-cache memory** by sharing key/value heads across
  query heads, directly shrinking the `num_kv_heads` term in the same formula -- the mechanism this
  notebook's extension quantifies.